In [232]:
import numpy as np
import pickle

In [233]:
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def has_hook_error(z, x):
    hook_error_locations = []
    for row in z:
        for twindow in range(1, 4):
            mask = row > twindow
            xsyndrome = np.zeros((x.shape[0], x.shape[1]), dtype=bool)
            for j in range(x.shape[0]):
                masked = (x[j] > 0) & mask
                xsyndrome[j] = masked
            # A "hook error" occurs if for this twindow, no X-check covers only one qubit in the masked locations (i.e., all have 0 or >1 overlap)
            fallthrough = True
            for j in range(xsyndrome.shape[0]):
                if np.sum(xsyndrome[j]) == 1 or np.sum(xsyndrome[j]) == 3:
                    fallthrough = False
                    break
            if fallthrough:
                hook_error_locations.append((row.copy(), twindow))
    return True if len(hook_error_locations) > 0 else False

def checkvalidschedule(schedule):
    for row in schedule:
        seen = set()
        for val in row:
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    for col_idx in range(schedule.shape[1]):
        seen = set()
        for row_idx in range(schedule.shape[0]):
            val = schedule[row_idx][col_idx]
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    return True

def randomize_schedule(schedule, max_attempts=1000):
    copyschedule = schedule.copy()
    if max_attempts == 0:
        return None
    for i in range(len(copyschedule)):
        nonzeros = list(range(1, max(copyschedule[i]) + 1)) if max(copyschedule[i]) != 1 else list(range(1, 5))
        np.random.shuffle(nonzeros)
        for j in range(len(copyschedule[i])):
            if copyschedule[i][j] != 0:
                copyschedule[i][j] = nonzeros.pop()
    if checkvalidschedule(copyschedule):
        return copyschedule, max_attempts
    else:
        return randomize_schedule(copyschedule, max_attempts - 1)


In [234]:
newzschedule, n = randomize_schedule(defaultzschedule, 1000)
print(1000 - n)
print(newzschedule)
newxschedule, n = randomize_schedule(defaultxschedule, 1000)
print(1000 - n)
print(newxschedule)


3
[[1 2 4 3 0 0 0 0]
 [0 0 0 0 1 2 3 4]
 [3 4 0 0 2 1 0 0]
 [4 0 1 0 3 0 2 0]]
0
[[3 8 2 7 5 4 1 6]]


In [235]:
arr = [1, 2, 3, 4, 0 ,0 , 0, 0]
nonzeros = list(range(1, max(arr) + 1))
print(nonzeros)
np.random.shuffle(nonzeros)
for i in range(len(arr)):
    if arr[i] != 0:
        arr[i] = nonzeros.pop()
print(arr)

[1, 2, 3, 4]
[4, 3, 2, 1, 0, 0, 0, 0]


In [2]:
singleschedules = pickle.load(open('singlepromotionschedules.pkl', 'rb'))
doubleschedules = pickle.load(open('doublepromotionschedules.pkl', 'rb'))

In [202]:
gauges = []
for schedule in singleschedules:
    stab = np.array(schedule, dtype = np.bool)[0]
    is_duplicate = False
    for g in gauges:
        if np.array_equal(stab, g) or np.array_equal(stab ^ np.array([1, 1, 1, 1, 1, 1, 1, 1]), g):
            is_duplicate = True
            break
    if is_duplicate:
        continue
    gauges.append(stab)
    print(stab.astype(int))
print(len(gauges))

[0 0 0 0 1 1 1 1]
[0 0 1 1 0 0 1 1]
[0 0 1 1 1 1 0 0]
[0 1 0 1 0 1 0 1]
[0 1 0 1 1 0 1 0]
[0 1 1 0 0 1 1 0]
[0 1 1 0 1 0 0 1]
7


In [ ]:
def get_schedule_id(schedule):
    index = 0
    id = 0
    for row in schedule:
        for val in row:
            id += val * (4 ** index)
            index += 1
    return id

In [ ]:
# Generate 10x10 schedules
# For single promotions, there are 7 gauges, and only 1 considered generator for each
schedulelist = [] # list of (z, x) schedules
hookerrorcount = 0
for gauge in gauges:
    xschedule = np.array([gauge.astype(int), (gauge ^ np.array([1, 1, 1, 1, 1, 1, 1, 1])).astype(int)])
    xschedules = []
    scheduleids = set()
    while len(xschedules) < 10:
        randxschedule, nx = randomize_schedule(xschedule, 1000)
        scheduleid = get_schedule_id(randxschedule)
        if scheduleid not in scheduleids:
            scheduleids.add(scheduleid)
            xschedules.append(randxschedule.copy())
    zschedules = []
    scheduleids = set()
    while len(zschedules) < 10:
        randzschedule, nz = randomize_schedule(defaultzschedule, 1000)
        scheduleid = get_schedule_id(randzschedule)
        if scheduleid not in scheduleids:
            scheduleids.add(scheduleid)
            zschedules.append(randzschedule)
    for z in zschedules:
        for x in xschedules:
            schedulelist.append((z, x))
            if has_hook_error(z, x):
                hookerrorcount += 1
print(len(schedulelist), hookerrorcount)

700 610


In [247]:
# Generate 100 random, independent (z, x) schedule pairs
# For single promotions, there are 7 gauges, and only 1 considered generator for each
schedulespergauge = 100
schedulelist = []  # list of (z, x) schedules
hookerrorexpected = np.zeros(schedulespergauge * len(gauges), dtype = np.bool)
for gauge_index, gauge in enumerate(gauges):
    xschedule = np.array([gauge.astype(int), (gauge ^ np.array([1, 1, 1, 1, 1, 1, 1, 1])).astype(int)])
    used_pairs = set()
    pairs_generated = 0
    while pairs_generated < schedulespergauge:
        randxschedule, nx = randomize_schedule(xschedule, 1000)
        randzschedule, nz = randomize_schedule(defaultzschedule, 1000)
        pair_id = (get_schedule_id(randxschedule), get_schedule_id(randzschedule))
        if pair_id in used_pairs:
            continue
        used_pairs.add(pair_id)
        schedulelist.append((randzschedule.copy(), randxschedule.copy()))
        if has_hook_error(randzschedule, randxschedule):
            hookerrorexpected[pairs_generated + schedulespergauge * gauge_index] = True
        pairs_generated += 1
print(len(schedulelist), np.sum(hookerrorexpected))

700 604


In [248]:
import pickle

with open('schedulelist.pkl', 'wb') as f:
    pickle.dump(schedulelist, f)

In [226]:
uniqueids = set()
gauge = gauges[0]
hookerrorcount = 0
for _ in range(100):
    xschedule = np.array([gauge.astype(int), (gauge ^ np.array([1, 1, 1, 1, 1, 1, 1, 1])).astype(int)])
    randxschedule, nx = randomize_schedule(xschedule, 1000)
    randzschedule, nz = randomize_schedule(defaultzschedule, 1000)
    if has_hook_error(randzschedule, randxschedule):
        hookerrorcount += 1
    scheduleid = get_schedule_id(randxschedule)
    uniqueids.add(scheduleid)
print(len(uniqueids), hookerrorcount)

93 100


In [113]:
print(randzschedule)
print(randxschedule)

[[3 4 1 2 0 0 0 0]
 [0 0 0 0 1 4 3 2]
 [2 1 0 0 4 3 0 0]
 [1 0 4 0 3 0 2 0]]
[[0 0 0 0 1 4 2 3]
 [3 2 1 4 0 0 0 0]]
